In [1]:
import pickle
import os
import numpy as np
import json
import kagglehub
import pandas as pd
from sklearn.decomposition import TruncatedSVD

DATA_DIR = os.path.join("..", "data")

In [2]:
with open(os.path.join(DATA_DIR, "als_model.pkl"), "rb") as f:
    model = pickle.load(f)
with open(os.path.join(DATA_DIR, "anilist_to_mal.json"), "r") as f:
    mal_ids = json.load(f)
with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))


In [3]:
anime_tag_vectors = np.load(os.path.join(DATA_DIR, "anime_tag_vectors.npy"))
manga_tag_vectors = np.load(os.path.join(DATA_DIR, "manga_tag_vectors.npy"))

als_user_factors = model.user_factors 
als_item_factors = model.item_factors 

In [4]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()}

# AniList ID -> real MAL ID (via idMal crosswalk; drop failed lookups)
anilist_to_mal = {int(k): v for k, v in mal_ids.items() if v is not None}

none_count = sum(1 for v in mal_ids.values() if v is None)
print(f"anilist_to_mal: {none_count}/{len(mal_ids)} AniList entries had no MAL match (idMal was null) -- dropped")

# real MAL ID -> ratings dataset's internal animeID (bridge step that was missing before)
animes['mal_id'] = animes['mal_url'].str.extract(r'/anime/(\d+)').astype(int)

# --- sanity-check the bridge tables before trusting a dict built from them ---
dup_mal = animes['mal_id'].duplicated(keep=False)
dup_animeid = animes['animeID'].duplicated(keep=False)
if dup_mal.any():
    print(f"WARNING: {dup_mal.sum()} rows in animes share a duplicated mal_id -- "
          f"dict(zip(...)) will silently keep only the last row per key")
if dup_animeid.any():
    print(f"WARNING: {dup_animeid.sum()} rows in animes share a duplicated animeID")
if not dup_mal.any() and not dup_animeid.any():
    print("animes['mal_id'] and animes['animeID'] are both unique")

mal_to_animeid = dict(zip(animes['mal_id'], animes['animeID']))

# --- walk the full chain: AniList idx -> real MAL id -> dataset animeID -> ALS row ---
aligned_rows = []
for i, a in enumerate(anime_content):
    anilist_id = a['id']
    real_mal_id = anilist_to_mal.get(anilist_id)
    if real_mal_id is None:
        continue  # no MAL match for this AniList entry


    animeid = mal_to_animeid.get(real_mal_id)
    if animeid is None:
        continue  # MAL id doesn't appear in the ratings dataset at all

    als_row = anime_id_map_reverse.get(animeid)
    if als_row is None:
        continue  # in the ratings dataset, but never rated in `train` -> no ALS row

    aligned_rows.append({
        'anilist_idx': i,
        'real_mal_id': real_mal_id,
        'animeid': animeid,
        'als_row': als_row,
    })

aligned_df = pd.DataFrame(aligned_rows)
print(f"Aligned {len(aligned_df)} / {len(anime_content)} AniList anime through all three ID systems to an ALS row")


anilist_to_mal: 11/4950 AniList entries had no MAL match (idMal was null) -- dropped
animes['mal_id'] and animes['animeID'] are both unique
Aligned 4778 / 5000 AniList anime through all three ID systems to an ALS row


In [5]:
def anilist_title(a):
    # AniList title is a dict, not a plain string
    t = a['title']
    return t.get('english') or t.get('romaji') or t.get('native')

animes_title_by_id = animes.set_index('animeID')['title']

sample = aligned_df.sample(min(10, len(aligned_df)), random_state=2)
for _, row in sample.iterrows():
    anilist_t = anilist_title(anime_content[row['anilist_idx']])
    ratings_t = animes_title_by_id.loc[row['animeid']]
    print(f"AniList: {anilist_t!r:55} | animes: {ratings_t!r}")


AniList: 'Legend of the Galactic Heroes Gaiden: A Hundred Billion Stars' | animes: 'Legend of the Galactic Heroes Gaiden'
AniList: 'One Piece Special: Protect! The Last Great Performance' | animes: 'One Piece: Protect! The Last Great Performance'
AniList: 'Higehiro: After Being Rejected, I Shaved and Took in a High School Runaway' | animes: 'Higehiro: After Being Rejected, I Shaved and Took in a High School Runaway'
AniList: 'That Time I Got Reincarnated as a Slime Season 3'      | animes: 'That Time I Got Reincarnated as a Slime Season 3'
AniList: 'Detective Conan: The Scarlet Bullet'                   | animes: 'Detective Conan Movie 24: The Scarlet Bullet'
AniList: 'Cross Ange: Rondo of Angel and Dragon'                 | animes: 'Cross Ange: Rondo of Angel and Dragon'
AniList: 'Durarara!! Specials'                                   | animes: 'Durarara!! Specials'
AniList: 'Kaze no Stigma'                                        | animes: 'Kaze no Stigma'
AniList: 'BASTARD!! -Heavy M

In [6]:
for n in [64, 128, 200, 300]:
    svd_test = TruncatedSVD(n_components=n, random_state=42)
    svd_test.fit(anime_tag_vectors)
    print(f"{n} dims -> {svd_test.explained_variance_ratio_.sum():.2%} variance explained")

64 dims -> 44.10% variance explained
128 dims -> 65.04% variance explained
200 dims -> 81.43% variance explained
300 dims -> 95.03% variance explained


In [7]:
combined_item_features = []
for _, row in aligned_df.iterrows():
    als_vec = als_item_factors[row['als_row']]           
    tag_vec = anime_tag_vectors[row['anilist_idx']]   
    combined = np.concatenate([als_vec, tag_vec])     
    combined_item_features.append(combined)

combined_item_features = np.array(combined_item_features)

aligned_animeids = set(aligned_df['animeid'])

train_aligned = train[train['anime_id'].isin(aligned_animeids)]

print(f"Original train rows: {len(train):,}")
print(f"Filtered to aligned anime: {len(train_aligned):,}")
print(f"Unique users remaining: {train_aligned['user_id'].nunique():,}")

Original train rows: 118,387,327
Filtered to aligned anime: 112,708,907
Unique users remaining: 1,773,734


In [8]:
positive_counts = train_aligned[train_aligned['is_positive'] == 1].groupby('user_id').size()
qualifying_users = positive_counts[positive_counts >= 5].index
train_final = train_aligned[train_aligned['user_id'].isin(qualifying_users)]

print(f"Final training rows: {len(train_final):,}")
print(f"Final unique users: {train_final['user_id'].nunique():,}")

Final training rows: 110,749,861
Final unique users: 1,461,421


In [9]:
MAX_POSITIVES_PER_USER = 50

train_positive = train_final[train_final['is_positive'] == 1]
train_positive_shuffled = train_positive.sample(frac=1, random_state=42)
train_capped = train_positive_shuffled.groupby('user_id').head(MAX_POSITIVES_PER_USER)

print(f"Training pairs after capping: {len(train_capped):,}")

Training pairs after capping: 39,179,955


In [10]:
animeid_to_aligned_idx = {row['animeid']: i for i, row in aligned_df.reset_index(drop=True).iterrows()}
user_id_to_useridx = train_final[['user_id', 'user_idx']].drop_duplicates().set_index('user_id')['user_idx'].to_dict()

print(len(animeid_to_aligned_idx), len(user_id_to_useridx))

4730 1461421


In [12]:
import random
import torch
from torch.utils.data import Dataset

class TwoTowerDataset(Dataset):
    def __init__(self, train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids):
        # Only keep rows where the anime is actually in our aligned set
        # (should already be true given train_final's filtering, but worth being defensive)
        self.data = train_capped.reset_index(drop=True)
        self.animeid_to_aligned_idx = animeid_to_aligned_idx
        self.user_id_to_useridx = user_id_to_useridx
        self.aligned_animeids = list(aligned_animeids)  # for fast random.choice sampling

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        user_idx = self.user_id_to_useridx[row['user_id']]
        pos_anime_idx = self.animeid_to_aligned_idx[row['anime_id']]

        # Sample a negative anime at random from the aligned set
        neg_anime_id = random.choice(self.aligned_animeids)
        neg_anime_idx = self.animeid_to_aligned_idx[neg_anime_id]

        return {
            'user_idx': user_idx,
            'pos_item_idx': pos_anime_idx,
            'neg_item_idx': neg_anime_idx,
        }

In [13]:
aligned_animeids = set(aligned_df['animeid'])

dataset = TwoTowerDataset(train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids)

print(len(dataset))
sample = dataset[0]
print(sample)

39179955
{'user_idx': 1253493, 'pos_item_idx': 836, 'neg_item_idx': 4651}
